# Detecting a Periodic Signal in a Quasar Light Curve with a Gaussian Process

This notebook searches the g-band light curve for a periodic modulation with a Lomb-Scargle periodogram, quantifies its significance, models the stochastic variability with a smooth Gaussian process, and validates the model with cross-validation.

**Set your NetID below to load your dataset.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scipy.optimize import minimize
from scipy.linalg import cho_factor, cho_solve
from astropy.timeseries import LombScargle

NETID = 'yournetid'  # <-- set this
d = np.genfromtxt(f'data/{NETID}.csv', delimiter=',', names=True)
t, y, e = d['t_days'], d['mag_g'], d['sigma_mag']
T = t.max() - t.min()
print(f"Loaded {t.size} epochs spanning {T:.0f} days")

## Step 1: Period search

The Lomb-Scargle periodogram is the standard tool for unevenly sampled data. With a dense cadence (median spacing of a few days within each season) periods from 20 days up to half the baseline are well sampled, so we search that whole range on a fine frequency grid and take the tallest peak. Its significance is the false-alarm probability.

In [ ]:
freq = np.linspace(1. / (T / 2.), 1. / 20., 6000)
ls = LombScargle(t, y, e)
power = ls.power(freq)
k = np.argmax(power)
P_best, p_best = 1. / freq[k], power[k]
fap = ls.false_alarm_probability(p_best, minimum_frequency=freq[0], maximum_frequency=freq[-1])

# period uncertainty from the peak (standard result for a sinusoid in white noise)
th = ls.model_parameters(freq[k])
amp = np.hypot(th[1], th[2])
resid = y - ls.model(t, freq[k])
sig_f = np.sqrt(6. / t.size) * resid.std() / (np.pi * T * amp)
P_err = P_best ** 2 * sig_f

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(1. / freq, power, lw=0.8); ax[0].axvline(P_best, color='C3', ls='--')
ax[0].set_xscale('log'); ax[0].set_xlabel('period (days)'); ax[0].set_ylabel('LS power'); ax[0].set_title('Lomb-Scargle periodogram')
ph = (t / P_best) % 1
ax[1].errorbar(ph, y, e, fmt='.', ms=3, lw=0.5); ax[1].invert_yaxis()
ax[1].set_xlabel('phase'); ax[1].set_ylabel('g (mag)'); ax[1].set_title(f'folded on P = {P_best:.1f} d')
plt.tight_layout(); plt.show()
print(f"Tallest peak: P = {P_best:.1f} +/- {P_err:.1f} d, power {p_best:.3f}, false-alarm probability {fap:.1e}")

In [ ]:
display(Markdown(f"The periodogram has a single dominant peak at **P = {P_best:.1f} ± {P_err:.1f} d**. Its false-alarm probability is **{fap:.0e}**, many orders of magnitude below any conventional threshold, so the periodicity is highly significant. The folded light curve shows the expected coherent modulation. This quasar is a periodic candidate."))

## Step 2: Gaussian-process model of the variability

Quasar variability on these time scales is smooth, so we model it with a squared-exponential kernel, $k(\Delta t) = \sigma^2 \exp(-\Delta t^2 / 2\ell^2)$, which is infinitely differentiable and the standard default choice. We maximize the marginal likelihood over the mean, amplitude $\sigma$ and length scale $\ell$.

In [ ]:
def nll(p, tt=t, yy=y, ee=e):
    mu, ls_, ll = p
    K = np.exp(2 * ls_) * np.exp(-0.5 * ((tt[:, None] - tt[None, :]) / np.exp(ll)) ** 2) + np.diag(ee ** 2)
    try:
        c = cho_factor(K, lower=True)
    except np.linalg.LinAlgError:
        return 1e10
    r = yy - mu
    return 0.5 * r @ cho_solve(c, r) + np.log(np.diag(c[0])).sum() + 0.5 * yy.size * np.log(2 * np.pi)

p0 = [y.mean(), np.log(y.std()), np.log(30.)]
res = minimize(nll, p0, method='Nelder-Mead', options=dict(xatol=1e-4, fatol=1e-6, maxiter=1500))
mu_hat, sig_hat, ell_hat = res.x[0], np.exp(res.x[1]), np.exp(res.x[2])

# error on ell from the curvature of the likelihood at the maximum
h = 0.05
f0 = lambda dl: nll([res.x[0], res.x[1], res.x[2] + dl])
curv = (f0(h) - 2 * f0(0.) + f0(-h)) / h ** 2
ell_err = ell_hat / np.sqrt(curv)
print(f"MAP: sigma = {sig_hat:.3f} mag, ell = {ell_hat:.1f} +/- {ell_err:.1f} d, ln L = {-res.fun:.1f}")

## Step 3: Validation

We check the model with 5-fold cross-validation: hold out a fifth of the epochs, predict them from the rest, and compare the predictions with the data. The GP's 68% predictive band (including the measurement noise) should contain about 68% of the held-out points.

In [ ]:
def predict(ttr, ytr, etr, tte, ete, sig, ell, mu):
    K = sig ** 2 * np.exp(-0.5 * ((ttr[:, None] - ttr[None, :]) / ell) ** 2) + np.diag(etr ** 2)
    c = cho_factor(K, lower=True)
    Ks = sig ** 2 * np.exp(-0.5 * ((tte[:, None] - ttr[None, :]) / ell) ** 2)
    m = mu + Ks @ cho_solve(c, ytr - mu)
    v = sig ** 2 - np.einsum('ij,ji->i', Ks, cho_solve(c, Ks.T)) + ete ** 2
    return m, np.sqrt(np.clip(v, 1e-12, None))

rng = np.random.default_rng(1)
folds = rng.permutation(t.size) % 5
mu_cv = np.empty(t.size); sd_cv = np.empty(t.size)
for kf in range(5):
    te = folds == kf
    mu_cv[te], sd_cv[te] = predict(t[~te], y[~te], e[~te], t[te], e[te], sig_hat, ell_hat, mu_hat)
r2 = 1 - np.sum((y - mu_cv) ** 2) / np.sum((y - y.mean()) ** 2)
cover = np.mean(np.abs(y - mu_cv) < sd_cv)

tg = np.linspace(t.min(), t.max(), 800)
mg, sg = predict(t, y, e, tg, np.zeros_like(tg), sig_hat, ell_hat, mu_hat)
plt.figure(figsize=(11, 3.6))
plt.errorbar(t, y, e, fmt='k.', ms=3, lw=0.5)
plt.plot(tg, mg, 'C0'); plt.fill_between(tg, mg - sg, mg + sg, color='C0', alpha=0.3)
plt.gca().invert_yaxis(); plt.xlabel('t (days)'); plt.ylabel('g (mag)'); plt.title('GP posterior mean and 68% band (squared-exponential kernel)')
plt.tight_layout(); plt.show()
print(f"5-fold CV: R^2 = {r2:.3f}, 68% band contains {100 * cover:.0f}% of the held-out points")

In [ ]:
display(Markdown(f"The Gaussian process describes the variability well: **5-fold cross-validated R² = {r2:.2f}**, and its 68% band contains {100 * cover:.0f}% of the held-out epochs. The model is validated. The characteristic variability time scale is **ℓ = {ell_hat:.0f} ± {ell_err:.0f} d** and the rms amplitude is **{sig_hat:.2f} mag**.\n\n**Conclusion.** The light curve contains a statistically significant periodic signal at P = {P_best:.0f} ± {P_err:.0f} d (false-alarm probability {fap:.0e}) on top of smooth stochastic variability that a squared-exponential Gaussian process describes with a validated predictive accuracy. We recommend this quasar as a supermassive black-hole binary candidate."))